# ResNet50 U-Net with the DEM + EvaNet's regions

Same model, loss, augmentation, epochs and cutoff procedure as `unet_dem.ipynb`. With `USE_EVANET_REGIONS = True`, training also gets EvaNet's 3 North Carolina regions (Hurricane Matthew 2016, loaded by `scripts/evanet_data.py`). Val (image 12) and test (4, 5, 6, 7, 10, 11) are unchanged.

Differences from `unet_dem.ipynb`, needed to fit the larger training set in memory:
- Tiles are preprocessed **in place**, and training tiles are fed from a generator with a full shuffle each epoch instead of `from_tensor_slices`, which made extra copies of the whole training set.
- The project and region tiles stay as two arrays rather than being concatenated.

Run with `USE_EVANET_REGIONS = False` for a baseline that uses this same pipeline on this machine.

Weights go to `../outputs/<RUN_NAME>.weights.h5` and results to `../outputs/<RUN_NAME>_results.json`, so `unet_dem.weights.h5` is never overwritten.

**Kernel:** the TensorFlow environment (`tflabsfall24`).

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras

sys.path.append("../scripts")
from dataset import get_tiles, SPLITS
from evanet_data import get_region_tiles

SEED = 0
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
USE_EVANET_REGIONS = True  # False = baseline with this same pipeline
TRAIN_SPLIT = "train"      # "train" = 1, 2, 3, 8, 9; "train_plus_new" = also 13-17 and 19
MAX_UNKNOWN = 1.0          # same as unet_dem.ipynb: keep every training tile
EPOCHS = 15                # same as unet_dem.ipynb

RUN_NAME = (("unet_dem_regions" if USE_EVANET_REGIONS else "unet_dem_rerun")
            + ("_plusnew" if TRAIN_SPLIT == "train_plus_new" else ""))
WEIGHTS = Path(f"../outputs/{RUN_NAME}.weights.h5")
RESULTS = Path(f"../outputs/{RUN_NAME}_results.json")
print("run:", RUN_NAME)

In [ ]:
# Training data is a list of (X, y) arrays, kept separate so nothing is copied
train_parts = [get_tiles(SPLITS[TRAIN_SPLIT], stride=128, max_unknown=MAX_UNKNOWN)]
if USE_EVANET_REGIONS:
    train_parts.append(get_region_tiles(stride=128, max_unknown=MAX_UNKNOWN))
X_val, y_val = get_tiles(SPLITS["val"])
X_test, y_test = get_tiles(SPLITS["test"])

for name, X, y in [*[(f"Train{k}", *p) for k, p in enumerate(train_parts)], ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    known = y[y != -1]
    print(f"{name:6}  X {X.shape}  unknown {(y == -1).mean():.1%}  flood (of known) {known.mean():.1%}")
print(f"training split: {TRAIN_SPLIT} {SPLITS[TRAIN_SPLIT]}  regions: {USE_EVANET_REGIONS}  "
      f"tiles: {sum(len(X) for X, _ in train_parts)}")

## Prepare the images for ResNet50

Same as `unet_dem.ipynb`: RGB goes to 0-255 and then `preprocess_input`, and the per-image-centred DEM is multiplied by `DEM_SCALE = 20`. It's done in place in chunks, so no second copy of the data is made.

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

DEM_SCALE = 20  # same as unet_dem.ipynb


def preprocess_inplace(X, chunk=256):
    """Same result as unet_dem.ipynb's preprocess(), written back into X."""
    for s in range(0, len(X), chunk):
        c = X[s:s + chunk]
        c[..., :3] = preprocess_input(c[..., :3] * 255.0)
        c[..., 3] *= DEM_SCALE
    return X


for X, _ in train_parts:
    preprocess_inplace(X)
preprocess_inplace(X_val)
preprocess_inplace(X_test)

X0 = train_parts[0][0]
print("RGB min/max:", X0[..., :3].min(), X0[..., :3].max())
print("DEM min/max:", X0[..., 3].min(), X0[..., 3].max())

## Loss function (skips unknown pixels)

Same as `unet_dem.ipynb`.

In [ ]:
def masked_bce(y_true, y_pred):
    known = tf.cast(y_true != -1, "float32")
    y_true = tf.where(y_true == -1, 0.0, y_true)
    loss = keras.losses.binary_crossentropy(y_true, y_pred)[..., None]
    return tf.reduce_sum(loss * known) / tf.reduce_sum(known)

## Build the model: ResNet50 encoder

Same as `unet_dem.ipynb` (from https://github.com/nikhilroxtomar/Semantic-Segmentation-Architecture/blob/main/TensorFlow/resnet50_unet.py): ImageNet weights, plus a 4th DEM input to `conv1` that starts at zero.

In [ ]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Conv2DTranspose, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50


def conv_block(x, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def decoder_block(x, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, 2, strides=2, padding="same")(x)
    x = Concatenate()([x, skip_features])
    return conv_block(x, num_filters)


def build_resnet50_unet(input_shape):
    inputs = Input(input_shape)
    resnet50 = ResNet50(include_top=False, weights=None, input_tensor=inputs)

    pretrained = ResNet50(include_top=False, weights="imagenet", input_shape=(256, 256, 3))
    for layer, src in zip(resnet50.layers, pretrained.layers):
        weights = src.get_weights()
        if layer.name == "conv1_conv":
            kernel, bias = weights
            weights = [np.concatenate([kernel, np.zeros_like(kernel[:, :, :1])], axis=2), bias]
        layer.set_weights(weights)

    s1 = inputs
    s2 = resnet50.get_layer("conv1_relu").output
    s3 = resnet50.get_layer("conv2_block3_out").output
    s4 = resnet50.get_layer("conv3_block4_out").output
    b1 = resnet50.get_layer("conv4_block6_out").output

    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    return Model(inputs, outputs, name="ResNet50_U-Net"), resnet50


model, encoder = build_resnet50_unet((256, 256, 4))
print(model.input_shape, "->", model.output_shape)

## Train

Same settings as `unet_dem.ipynb`: frozen encoder except `conv1`, Adam 1e-3, batch 4, 15 epochs, random rot90 and flip, and the best `val_loss` epoch is saved. Training tiles come from a generator that reshuffles every tile each epoch.

In [ ]:
encoder.trainable = False
encoder.get_layer("conv1_conv").trainable = True

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=masked_bce, jit_compile=False)

WEIGHTS.parent.mkdir(exist_ok=True)
checkpoint = keras.callbacks.ModelCheckpoint(str(WEIGHTS), monitor="val_loss",
                                             save_best_only=True, save_weights_only=True)


def augment(x, y):
    """Same as unet_dem.ipynb: random rot90, then a 50% left-right flip (tf.cond instead of a Python if)."""
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x, y = tf.image.rot90(x, k), tf.image.rot90(y, k)
    flip = tf.random.uniform([]) < 0.5
    x = tf.cond(flip, lambda: tf.image.flip_left_right(x), lambda: x)
    y = tf.cond(flip, lambda: tf.image.flip_left_right(y), lambda: y)
    return x, y


index = [(p, i) for p, (X, _) in enumerate(train_parts) for i in range(len(X))]
rng = np.random.default_rng(SEED)


def train_tiles():
    for j in rng.permutation(len(index)):
        p, i = index[j]
        X, y = train_parts[p]
        yield X[i], y[i].astype(np.float32)[..., None]


signature = (tf.TensorSpec((256, 256, 4), tf.float32), tf.TensorSpec((256, 256, 1), tf.float32))
train_ds = (tf.data.Dataset.from_generator(train_tiles, output_signature=signature)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(4).prefetch(2))


def val_tiles():
    for i in range(len(X_val)):
        yield X_val[i], y_val[i].astype(np.float32)[..., None]


val_ds = tf.data.Dataset.from_generator(val_tiles, output_signature=signature).batch(4)

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint], verbose=2)

## Results

Same metrics as `unet_dem.ipynb`: flood and dry IoU over known pixels and the share predicted flood. The cutoff is picked on **val** from 0.3-0.7 by the mean of flood and dry IoU (`unet_dem` picked 0.4 by hand from the same sweep), and test is scored at that cutoff and at 0.5.

In [ ]:
model.load_weights(str(WEIGHTS))


def scores(probs, y, cut):
    """Flood IoU, dry IoU, share of known pixels predicted flood, share labelled flood."""
    pf, k = probs > cut, y != -1
    tf_, td = y == 1, y == 0
    f = np.sum(pf & tf_ & k) / np.sum((pf | tf_) & k)
    d = np.sum(~pf & td & k) / np.sum((~pf | td) & k)
    return float(f), float(d), float(pf[k].mean()), float(tf_[k].mean())


val_probs = model.predict(X_val, batch_size=8, verbose=0)[..., 0]
sweep = {cut: scores(val_probs, y_val, cut) for cut in [0.3, 0.4, 0.5, 0.6, 0.7]}
for cut, (f, d, s, _) in sweep.items():
    print(f"val cutoff {cut}: flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {s:.1%}")
CUTOFF = max(sweep, key=lambda c: sweep[c][0] + sweep[c][1])
print("chosen cutoff:", CUTOFF)

test_probs = model.predict(X_test, batch_size=8, verbose=0)[..., 0]
results = {"run": RUN_NAME, "cutoff": CUTOFF, "use_evanet_regions": USE_EVANET_REGIONS,
           "train_split": TRAIN_SPLIT, "train_images": SPLITS[TRAIN_SPLIT],
           "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
           "val": {}, "test": {}, "per_image": {}}
# Per-image: the test images, then the held-out Ida scene (never trained, validated or tuned on)
per_image_probs = {}
for i in SPLITS["test"] + SPLITS["ida_hard_case"]:
    Xi, yi = get_tiles([i])
    per_image_probs[i] = (model.predict(preprocess_inplace(Xi), batch_size=8, verbose=0)[..., 0], yi)

for cut in sorted({CUTOFF, 0.5}):
    results["val"][cut] = sweep[cut]
    results["test"][cut] = scores(test_probs, y_test, cut)
    f, d, s, l = results["test"][cut]
    print(f"\nTEST cutoff {cut}: flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {s:.1%}  label {l:.1%}")
    for i, (pi, yi) in per_image_probs.items():
        results["per_image"].setdefault(str(i), {})[cut] = scores(pi, yi, cut)
        f, d, s, l = results["per_image"][str(i)][cut]
        tag = "  (Ida hard case)" if i in SPLITS["ida_hard_case"] else ""
        print(f"  image {i:2}: flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {s:.1%}  label {l:.1%}{tag}")

RESULTS.write_text(json.dumps(results, indent=1))
print("\nsaved", RESULTS)